[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/A.%20Foundational%20Analytics%20%26%20Inventory%20Control%20%28The%20Language%20of%20Supply%20Chain%29/048.%20The%20Demand%20Forecasting%20-%20Exponential%20Smoothing%20Problem/P48-Tier-4_executed.ipynb) [![Open In SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open In Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---


# 48. Demand Forecasting: Exponential Smoothing

## Tier 4: A Neural Forecasting Augmentation

### Purpose and scope

This notebook demonstrates a small neural network that predicts the next demand value from a fixed-length history window. It also produces an auxiliary scalar called `alpha` and a simple uncertainty interval. The example compares its forecasts with simple and heuristic adaptive exponential-smoothing baselines.

The implementation is a compact NumPy teaching example, not a full self-supervised learning system. Training uses observed next-period demand targets, so it is supervised one-step forecasting. The `alpha` output is regularized toward a chosen target, but it is not used to update the forecast and therefore is not an estimated exponential-smoothing coefficient. The attention calculation receives a single sequence position in this implementation; its weight is consequently always one and does not discover temporal patterns. The uncertainty interval is a fixed-width heuristic, not a learned or calibrated confidence interval.

### Inputs, outputs, and assumptions

- **Input:** A chronological series of weekly container demand, measured in TEU.
- **Forecast input:** The most recent 20 observations, transformed using statistics fitted to training data.
- **Outputs:** Recursive multi-step point forecasts, an auxiliary `alpha` value, attention weights, and heuristic interval bounds.
- **Training:** Sliding windows are divided chronologically into training and validation sets. The held-out final 20 weeks are reserved for the final comparison.
- **Scope:** The synthetic series combines a piecewise trend, recurring seasonal multipliers, and random noise. It does not represent measured port data. The model does not include external predictors, a probabilistic likelihood, a calibrated interval model, or a full multi-head attention mechanism.

### Method and mathematical objective

A sliding window of length $L$ turns a demand series $d_1,\ldots,d_T$ into input-target pairs. For window position $t$, the input is $\mathbf{x}_t=(d_t,\ldots,d_{t+L-1})$ and the target is $y_t=d_{t+L}$. A neural network maps the normalized input to a forecast $\hat y_t$ and an auxiliary value $a_t\in(0,1)$.

The training objective combines squared forecast error with a small penalty that keeps the auxiliary output near a reference value $a_0=0.3$:

$$\mathcal{L}=\frac{1}{N}\sum_{t=1}^{N}\left[(\hat y_t-y_t)^2+0.1(a_t-a_0)^2\right].$$

Here, $N$ is the number of training examples. The first term trains next-period forecasts; the second regularizes the auxiliary output. It does not make $a_t$ part of the forecast equation. ReLU activations are used in the hidden layers, and a sigmoid constrains $a_t$ to $(0,1)$. Backpropagation updates the weights by gradient descent.

For a multi-step forecast, the model predicts one period ahead, appends that prediction to the input history, and repeats. Thus later predictions depend on earlier model predictions, rather than on unknown future observations.

The displayed interval is $[\hat y-h,\hat y+h]$, where $h=1.96(0.1s_y)$ and $s_y$ is the standard deviation of training targets. The factor 1.96 resembles a normal 95% interval multiplier, but the fixed width and unverified distributional assumptions mean that the interval is only a heuristic. Its empirical coverage must not be interpreted as a guaranteed 95% confidence level.

### How to interpret the comparison

MAE and RMSE measure forecast error in TEU; MAPE expresses absolute error relative to observed demand; bias is the mean signed error (actual minus forecast). Lower MAE, RMSE, and MAPE indicate smaller errors, while bias near zero indicates little average over- or under-forecasting. Results apply only to this synthetic series and its chosen chronological holdout. A single experiment does not establish that one method generally outperforms another.


In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple, Union

import matplotlib.pyplot as plt
import numpy as np

plt.style.use("seaborn-v0_8")
np.random.seed(42)


@dataclass
class NeuralESConfig:
    """Settings for the compact neural next-step forecasting model."""

    input_window: int = 20
    hidden_size: int = 128
    learning_rate: float = 0.001
    epochs: int = 200
    batch_size: int = 32
    alpha_min: float = 0.05
    alpha_max: float = 0.95
    uncertainty_quantiles: Optional[List[float]] = None

    def __post_init__(self) -> None:
        if self.uncertainty_quantiles is None:
            self.uncertainty_quantiles = [0.025, 0.5, 0.975]


class SimpleNeuralNetwork:
    """A two-hidden-layer network trained with a manual NumPy backward pass."""

    def __init__(self, input_size: int, hidden_size: int, output_size: int):
        self.W1 = np.random.randn(input_size, hidden_size) * np.sqrt(2.0 / input_size)
        self.b1 = np.zeros((1, hidden_size))
        self.W2 = np.random.randn(hidden_size, hidden_size) * np.sqrt(2.0 / hidden_size)
        self.b2 = np.zeros((1, hidden_size))
        self.W3 = np.random.randn(hidden_size, output_size) * np.sqrt(2.0 / hidden_size)
        self.b3 = np.zeros((1, output_size))
        self.cache: Dict[str, np.ndarray] = {}

    @staticmethod
    def relu(x: np.ndarray) -> np.ndarray:
        return np.maximum(0.0, x)

    @staticmethod
    def relu_derivative(x: np.ndarray) -> np.ndarray:
        return (x > 0).astype(float)

    @staticmethod
    def sigmoid(x: np.ndarray) -> np.ndarray:
        return 1.0 / (1.0 + np.exp(-np.clip(x, -500, 500)))

    def attention_mechanism(
        self, x: np.ndarray
    ) -> Tuple[np.ndarray, np.ndarray]:
        """Return singleton-position weights and the unchanged input."""
        batch_size = x.shape[0]
        weights = np.ones((batch_size, 1, 1), dtype=float)
        return weights, x

    def forward(self, x: np.ndarray) -> Dict[str, np.ndarray]:
        if x.ndim != 2 or x.shape[1] != self.W1.shape[0]:
            raise ValueError("Input must have shape (batch_size, input_size).")

        x_sequence = x[:, np.newaxis, :]
        attention_weights, weighted_input = self.attention_mechanism(x_sequence)
        weighted_input = weighted_input[:, 0, :]

        z1 = weighted_input @ self.W1 + self.b1
        a1 = self.relu(z1)
        z2 = a1 @ self.W2 + self.b2
        a2 = self.relu(z2)
        z3 = a2 @ self.W3 + self.b3

        forecast = z3[:, 0:1]
        alpha = self.sigmoid(z3[:, 1:2])
        self.cache = {
            "weighted_input": weighted_input,
            "z1": z1,
            "a1": a1,
            "z2": z2,
            "a2": a2,
            "z3": z3,
            "alpha": alpha,
        }
        return {
            "forecast": forecast,
            "alpha": alpha,
            "attention_weights": attention_weights,
        }

    def backward(
        self,
        grad_forecast: np.ndarray,
        grad_alpha: np.ndarray,
        learning_rate: float = 0.001,
    ) -> None:
        if not self.cache:
            raise RuntimeError("Call forward before backward.")

        batch_size = grad_forecast.shape[0]
        if grad_forecast.shape != (batch_size, 1) or grad_alpha.shape != (batch_size, 1):
            raise ValueError("Output gradients must both have shape (batch_size, 1).")
        if self.cache["z3"].shape[0] != batch_size:
            raise ValueError("Gradient batch size must match the forward-pass batch.")

        dz3 = np.zeros_like(self.cache["z3"])
        dz3[:, 0:1] = grad_forecast
        alpha = self.cache["alpha"]
        dz3[:, 1:2] = grad_alpha * alpha * (1.0 - alpha)

        dW3 = self.cache["a2"].T @ dz3 / batch_size
        db3 = np.sum(dz3, axis=0, keepdims=True) / batch_size
        dz2 = (dz3 @ self.W3.T) * self.relu_derivative(self.cache["z2"])
        dW2 = self.cache["a1"].T @ dz2 / batch_size
        db2 = np.sum(dz2, axis=0, keepdims=True) / batch_size
        dz1 = (dz2 @ self.W2.T) * self.relu_derivative(self.cache["z1"])
        dW1 = self.cache["weighted_input"].T @ dz1 / batch_size
        db1 = np.sum(dz1, axis=0, keepdims=True) / batch_size

        self.W3 -= learning_rate * dW3
        self.b3 -= learning_rate * db3
        self.W2 -= learning_rate * dW2
        self.b2 -= learning_rate * db2
        self.W1 -= learning_rate * dW1
        self.b1 -= learning_rate * db1
        self.cache = {}


In [ ]:
class SelfSupervisedNeuralES:
    """Neural next-step forecaster with an auxiliary regularized alpha output."""

    def __init__(self, config: NeuralESConfig):
        self.config = config
        self.forecast_network = SimpleNeuralNetwork(
            input_size=config.input_window,
            hidden_size=config.hidden_size,
            output_size=2,
        )
        self.training_history: Dict[str, List[float]] = {
            "loss": [],
            "forecast_loss": [],
            "alpha_loss": [],
            "uncertainty_loss": [],
            "learned_alphas": [],
        }
        self.is_trained = False
        self.normalization_params: Optional[Dict[str, np.ndarray]] = None

    def prepare_training_data(
        self, demand: List[float]
    ) -> Tuple[np.ndarray, np.ndarray]:
        """Create chronological input windows and their next-period targets."""
        values = np.asarray(demand, dtype=float)
        if values.ndim != 1 or len(values) <= self.config.input_window:
            raise ValueError(
                f"Provide a one-dimensional series with more than "
                f"{self.config.input_window} observations."
            )
        X = np.array(
            [values[i : i + self.config.input_window]
             for i in range(len(values) - self.config.input_window)]
        )
        y = values[self.config.input_window :]
        return X, y

    def normalize_data(
        self, X: np.ndarray, y: np.ndarray
    ) -> Tuple[np.ndarray, np.ndarray, Dict[str, np.ndarray]]:
        """Fit normalization statistics on the supplied training partition."""
        params = {
            "X_mean": np.mean(X, axis=0),
            "X_std": np.std(X, axis=0) + 1e-8,
            "y_mean": np.asarray(np.mean(y)),
            "y_std": np.asarray(np.std(y) + 1e-8),
        }
        X_norm = (X - params["X_mean"]) / params["X_std"]
        y_norm = (y - params["y_mean"]) / params["y_std"]
        return X_norm, y_norm, params

    def self_supervised_loss(
        self, X: np.ndarray, y: np.ndarray
    ) -> Tuple[float, np.ndarray, np.ndarray]:
        """Compute next-step squared error and auxiliary-alpha regularization."""
        outputs = self.forecast_network.forward(X)
        forecast_error = outputs["forecast"] - y.reshape(-1, 1)
        alpha_error = outputs["alpha"] - 0.3
        forecast_loss = float(np.mean(forecast_error**2))
        alpha_loss = float(0.1 * np.mean(alpha_error**2))
        grad_forecast = 2.0 * forecast_error
        grad_alpha = 0.2 * alpha_error
        return forecast_loss + alpha_loss, grad_forecast, grad_alpha

    def train(
        self, demand: List[float], validation_split: float = 0.2
    ) -> Dict[str, List[float]]:
        """Train in chronological mini-batches and report validation loss."""
        if not 0.0 < validation_split < 1.0:
            raise ValueError("validation_split must be between 0 and 1.")
        X, y = self.prepare_training_data(demand)
        split_idx = int(len(X) * (1.0 - validation_split))
        if split_idx < 1 or split_idx >= len(X):
            raise ValueError("The data must provide nonempty train and validation sets.")

        X_train, y_train = X[:split_idx], y[:split_idx]
        X_val, y_val = X[split_idx:], y[split_idx:]
        X_train, y_train, self.normalization_params = self.normalize_data(
            X_train, y_train
        )
        X_val = (X_val - self.normalization_params["X_mean"]) / self.normalization_params["X_std"]
        y_val = (y_val - self.normalization_params["y_mean"]) / self.normalization_params["y_std"]

        print(f"Training samples: {len(X_train)}; validation samples: {len(X_val)}")
        self.training_history = {
            "loss": [],
            "forecast_loss": [],
            "alpha_loss": [],
            "uncertainty_loss": [],
            "learned_alphas": [],
        }

        for epoch in range(self.config.epochs):
            batch_losses = []
            batch_alphas = []
            for start in range(0, len(X_train), self.config.batch_size):
                end = min(start + self.config.batch_size, len(X_train))
                X_batch, y_batch = X_train[start:end], y_train[start:end]
                loss, grad_forecast, grad_alpha = self.self_supervised_loss(
                    X_batch, y_batch
                )
                batch_alphas.extend(
                    self.forecast_network.forward(X_batch)["alpha"].ravel().tolist()
                )
                self.forecast_network.backward(
                    grad_forecast, grad_alpha, self.config.learning_rate
                )
                batch_losses.append(loss)

            val_loss, _, _ = self.self_supervised_loss(X_val, y_val)
            train_loss = float(np.mean(batch_losses))
            mean_alpha = float(np.mean(batch_alphas))
            self.training_history["loss"].append(train_loss)
            self.training_history["forecast_loss"].append(val_loss)
            self.training_history["learned_alphas"].append(mean_alpha)
            if epoch % 20 == 0:
                print(
                    f"Epoch {epoch:3d}: train loss = {train_loss:.4f}, "
                    f"validation loss = {val_loss:.4f}, auxiliary alpha = {mean_alpha:.3f}"
                )

        self.is_trained = True
        print(
            "Training completed. Final validation loss: "
            f"{self.training_history['forecast_loss'][-1]:.4f}"
        )
        return self.training_history

    def forecast_with_uncertainty(
        self, demand: List[float], steps_ahead: int = 1
    ) -> Dict[str, Union[List[float], List[List[float]]]]:
        """Generate recursive forecasts and fixed-width heuristic intervals."""
        if not self.is_trained or self.normalization_params is None:
            raise ValueError("Train the model before forecasting.")
        if steps_ahead < 1:
            raise ValueError("steps_ahead must be at least 1.")
        if len(demand) < self.config.input_window:
            raise ValueError(
                f"Provide at least {self.config.input_window} historical observations."
            )

        params = self.normalization_params
        current_demand = list(map(float, demand))
        forecasts: List[float] = []
        learned_alphas: List[float] = []
        uncertainty_bounds: List[List[float]] = []
        attention_weights: List[List[float]] = []
        half_width = 1.96 * 0.1 * float(params["y_std"])

        for _ in range(steps_ahead):
            window = np.asarray(current_demand[-self.config.input_window:]).reshape(1, -1)
            X_norm = (window - params["X_mean"]) / params["X_std"]
            outputs = self.forecast_network.forward(X_norm)
            forecast_norm = float(outputs["forecast"][0, 0])
            forecast = forecast_norm * float(params["y_std"]) + float(params["y_mean"])
            alpha = float(
                np.clip(outputs["alpha"][0, 0], self.config.alpha_min, self.config.alpha_max)
            )
            forecasts.append(forecast)
            learned_alphas.append(alpha)
            uncertainty_bounds.append([forecast - half_width, forecast + half_width])
            attention_weights.append(outputs["attention_weights"][0, 0].tolist())
            current_demand.append(forecast)

        return {
            "forecasts": forecasts,
            "learned_alphas": learned_alphas,
            "uncertainty_bounds": uncertainty_bounds,
            "attention_weights": attention_weights,
        }


## Synthetic port-demand example

The following deterministic-seed example constructs 104 weekly observations. The final 20 weeks are held out for evaluation; all model fitting and normalization use only the earlier observations. The series is synthetic, so its patterns are useful for demonstrating the workflow but should not be treated as empirical evidence about port operations.

In [ ]:
np.random.seed(42)
weeks_104 = list(range(1, 105))

trend_component = []
for week in weeks_104:
    if week < 30:
        trend = 20000 + 40 * week
    elif week < 60:
        trend = 21200 + 20 * (week - 30)
    elif week < 80:
        trend = 21800 - 10 * (week - 60)
    else:
        trend = 21600 + 15 * (week - 80)
    trend_component.append(trend)

weekly_seasonal = ([1.0, 1.1, 1.2, 1.15, 1.05, 0.9, 0.8] * 15)[:104]
monthly_seasonal = []
for week in weeks_104:
    month_idx = (week - 1) % 52 // 4
    if month_idx < 3:
        factor = 1.15
    elif month_idx < 6:
        factor = 1.0
    elif month_idx < 9:
        factor = 0.85
    else:
        factor = 1.05
    monthly_seasonal.append(factor)

noise = []
for week in weeks_104:
    noise_std = 800 if week < 40 else 1200 if week < 70 else 600
    noise.append(np.random.normal(0, noise_std))

demand_104_weeks = [
    max(15000, int(trend * weekly * monthly + value))
    for trend, weekly, monthly, value in zip(
        trend_component, weekly_seasonal, monthly_seasonal, noise
    )
]

print("Synthetic port-container demand example")
print(f"Observations: {len(demand_104_weeks)} weekly values")
print(
    f"Demand range: {min(demand_104_weeks):,} to "
    f"{max(demand_104_weeks):,} TEU"
)
print(f"Mean demand: {np.mean(demand_104_weeks):,.0f} TEU")
print("Week | Demand (TEU) | Weekly factor | Seasonal factor | Trend | Noise")
print("-" * 76)
for i in range(0, 104, 13):
    print(
        f"{weeks_104[i]:4d} | {demand_104_weeks[i]:11d} | "
        f"{weekly_seasonal[i]:13.2f} | {monthly_seasonal[i]:15.2f} | "
        f"{trend_component[i]:6.0f} | {noise[i]:5.0f}"
    )

test_periods = 20
training_demand = demand_104_weeks[:-test_periods]
config = NeuralESConfig(
    input_window=20,
    hidden_size=128,
    learning_rate=0.001,
    epochs=200,
    batch_size=32,
    alpha_min=0.05,
    alpha_max=0.95,
)
neural_es = SelfSupervisedNeuralES(config)
training_history = neural_es.train(training_demand, validation_split=0.2)

print("\nTraining summary")
print(f"Final training loss: {training_history['loss'][-1]:.4f}")
print(f"Final validation loss: {training_history['forecast_loss'][-1]:.4f}")
print(f"Mean auxiliary alpha in final epoch: {training_history['learned_alphas'][-1]:.3f}")

## Forecast evaluation and baselines

Simple exponential smoothing (SES) updates its level after each observed value using $\ell_t=\alpha d_t+(1-\alpha)\ell_{t-1}$. When forecasting several periods without additional observations, the standard SES point forecast is the final estimated level at every horizon. The adaptive baseline below is intentionally a heuristic: it adjusts its coefficient using recent errors within the forecast recursion, where prior forecasts are used in place of unavailable future observations. It is included as an illustrative comparator, not as a validated adaptive forecasting procedure.

The neural network and both baselines receive the same pre-holdout observations. The actual values from the final 20 weeks are used only for evaluation.

In [ ]:
class SimpleES:
    """Simple exponential smoothing with a fixed smoothing coefficient."""

    def __init__(self, alpha: float = 0.3):
        if not 0.0 <= alpha <= 1.0:
            raise ValueError("alpha must be between 0 and 1.")
        self.alpha = alpha

    def forecast(self, demand: List[float], steps: int) -> List[float]:
        if not demand or steps < 1:
            raise ValueError("Provide demand observations and at least one step.")
        level = float(demand[0])
        for value in demand[1:]:
            level = self.alpha * float(value) + (1.0 - self.alpha) * level
        return [level] * steps


class AdaptiveES:
    """Heuristic alpha adjustment driven by recursively computed forecast errors."""

    def __init__(self, initial_alpha: float = 0.3):
        self.alpha = initial_alpha
        self.mad_window = 5

    def forecast(self, demand: List[float], steps: int) -> List[float]:
        if not demand or steps < 1:
            raise ValueError("Provide demand observations and at least one step.")
        level = float(demand[0])
        for value in demand[1:]:
            level = self.alpha * float(value) + (1.0 - self.alpha) * level

        forecasts: List[float] = []
        errors: List[float] = []
        previous_forecast = level
        for _ in range(steps):
            forecast = self.alpha * level + (1.0 - self.alpha) * previous_forecast
            error = abs(forecast - previous_forecast)
            errors.append(error)
            if len(errors) >= self.mad_window:
                mad = float(np.mean(errors[-self.mad_window:]))
                if mad > 2000:
                    self.alpha = min(0.95, self.alpha + 0.05)
                elif mad < 1000:
                    self.alpha = max(0.05, self.alpha - 0.02)
            forecasts.append(forecast)
            previous_forecast = forecast
        return forecasts


def calculate_metrics(
    actual: List[float], predicted: List[float], method_name: str
) -> Dict[str, Union[str, float]]:
    """Calculate standard point-forecast error summaries."""
    actual_array = np.asarray(actual, dtype=float)
    predicted_array = np.asarray(predicted, dtype=float)
    if actual_array.shape != predicted_array.shape or actual_array.size == 0:
        raise ValueError("Actual and predicted values must have equal nonzero length.")
    errors = actual_array - predicted_array
    nonzero = actual_array != 0
    mape = (
        float(np.mean(np.abs(errors[nonzero] / actual_array[nonzero])) * 100)
        if np.any(nonzero)
        else float("nan")
    )
    return {
        "method": method_name,
        "mae": float(np.mean(np.abs(errors))),
        "mse": float(np.mean(errors**2)),
        "rmse": float(np.sqrt(np.mean(errors**2))),
        "mape": mape,
        "bias": float(np.mean(errors)),
    }


neural_results = neural_es.forecast_with_uncertainty(training_demand, test_periods)
simple_forecasts = SimpleES(alpha=0.3).forecast(training_demand, test_periods)
adaptive_forecasts = AdaptiveES(initial_alpha=0.3).forecast(
    training_demand, test_periods
)
actual_values = demand_104_weeks[-test_periods:]

methods_comparison = [
    calculate_metrics(actual_values, simple_forecasts, "Simple ES"),
    calculate_metrics(actual_values, adaptive_forecasts, "Adaptive ES"),
    calculate_metrics(actual_values, neural_results["forecasts"], "Neural model"),
]

print("Forecast comparison on the final 20 weeks")
print(f"{'Method':<14} | {'MAE (TEU)':>10} | {'MAPE (%)':>9} | {'RMSE (TEU)':>11} | {'Bias (TEU)':>11}")
print("-" * 68)
for metrics in methods_comparison:
    print(
        f"{metrics['method']:<14} | {metrics['mae']:10.0f} | "
        f"{metrics['mape']:9.1f} | {metrics['rmse']:11.0f} | "
        f"{metrics['bias']:11.0f}"
    )

neural_metrics = methods_comparison[2]
simple_metrics = methods_comparison[0]
print("\nRelative error change for the neural model versus Simple ES")
for metric in ("mae", "mape", "rmse"):
    baseline = float(simple_metrics[metric])
    if baseline > 0:
        change = (baseline - float(neural_metrics[metric])) / baseline * 100
        print(f"{metric.upper()}: {change:+.1f}% improvement")

uncertainty_bounds = neural_results["uncertainty_bounds"]
coverage_count = sum(
    lower <= actual <= upper
    for actual, (lower, upper) in zip(actual_values, uncertainty_bounds)
)
coverage_percentage = coverage_count / len(actual_values) * 100
avg_interval_width = float(
    np.mean([upper - lower for lower, upper in uncertainty_bounds])
)
print("\nHeuristic interval diagnostics (not calibrated confidence guarantees)")
print(f"Observed holdout coverage: {coverage_percentage:.1f}%")
print(f"Mean interval width: {avg_interval_width:.0f} TEU")
print(
    f"Mean interval width relative to mean observed demand: "
    f"{avg_interval_width / np.mean(actual_values) * 100:.1f}%"
)

learned_alphas = neural_results["learned_alphas"]
print("\nAuxiliary alpha output (not used in the forecast equation)")
print(f"Mean: {np.mean(learned_alphas):.3f}")
print(f"Range: {min(learned_alphas):.3f} to {max(learned_alphas):.3f}")
print(f"Standard deviation: {np.std(learned_alphas):.3f}")

## Visual inspection

The first dashboard places training and validation losses, the auxiliary output, point forecasts, and heuristic intervals side by side. The second examines holdout errors, cumulative absolute error, the singleton attention weight, and a normalized comparison of selected metrics. These plots are diagnostic summaries of this one experiment; they do not establish general performance or interval calibration.

In [ ]:
epochs = np.arange(len(training_history["loss"]))
test_weeks = np.arange(
    len(demand_104_weeks) - test_periods + 1,
    len(demand_104_weeks) + 1,
)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Neural Forecasting: Training and Holdout Diagnostics", fontsize=16)

axes[0, 0].plot(epochs, training_history["loss"], label="Training loss")
axes[0, 0].plot(epochs, training_history["forecast_loss"], label="Validation loss")
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].set_ylabel("Loss (normalized scale)")
axes[0, 0].set_title("Training and validation loss")
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(epochs, training_history["learned_alphas"], label="Auxiliary alpha")
axes[0, 1].axhline(0.3, color="red", linestyle="--", label="Regularization target")
axes[0, 1].set_xlabel("Epoch")
axes[0, 1].set_ylabel("Auxiliary output")
axes[0, 1].set_title("Auxiliary output during training")
axes[0, 1].set_ylim(0, 1)
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(test_weeks, actual_values, "ko-", label="Actual")
axes[1, 0].plot(test_weeks, simple_forecasts, "r--", label="Simple ES")
axes[1, 0].plot(test_weeks, adaptive_forecasts, "b:", label="Heuristic adaptive ES")
axes[1, 0].plot(test_weeks, neural_results["forecasts"], "g-", label="Neural model")
axes[1, 0].set_xlabel("Week")
axes[1, 0].set_ylabel("Container volume (TEU)")
axes[1, 0].set_title("Final 20-week holdout forecasts")
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

lower_bounds = [bounds[0] for bounds in uncertainty_bounds]
upper_bounds = [bounds[1] for bounds in uncertainty_bounds]
axes[1, 1].plot(test_weeks, actual_values, "ko-", label="Actual")
axes[1, 1].plot(test_weeks, neural_results["forecasts"], "g-", label="Neural forecast")
axes[1, 1].fill_between(
    test_weeks, lower_bounds, upper_bounds, alpha=0.3,
    color="green", label="Heuristic interval"
)
axes[1, 1].set_xlabel("Week")
axes[1, 1].set_ylabel("Container volume (TEU)")
axes[1, 1].set_title("Fixed-width heuristic intervals")
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
simple_errors = np.asarray(actual_values) - np.asarray(simple_forecasts)
adaptive_errors = np.asarray(actual_values) - np.asarray(adaptive_forecasts)
neural_errors = np.asarray(actual_values) - np.asarray(neural_results["forecasts"])

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Additional Holdout Diagnostics", fontsize=16)

axes[0, 0].hist(simple_errors, bins=10, alpha=0.5, label="Simple ES", density=True)
axes[0, 0].hist(adaptive_errors, bins=10, alpha=0.5, label="Heuristic adaptive ES", density=True)
axes[0, 0].hist(neural_errors, bins=10, alpha=0.5, label="Neural model", density=True)
axes[0, 0].axvline(0, color="black", linewidth=0.8)
axes[0, 0].set_xlabel("Forecast error (TEU; actual minus forecast)")
axes[0, 0].set_ylabel("Density")
axes[0, 0].set_title("Holdout error distributions")
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(test_weeks, np.cumsum(np.abs(simple_errors)), label="Simple ES")
axes[0, 1].plot(test_weeks, np.cumsum(np.abs(adaptive_errors)), label="Heuristic adaptive ES")
axes[0, 1].plot(test_weeks, np.cumsum(np.abs(neural_errors)), label="Neural model")
axes[0, 1].set_xlabel("Week")
axes[0, 1].set_ylabel("Cumulative absolute error (TEU)")
axes[0, 1].set_title("Cumulative holdout error")
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

attention_sample = neural_results["attention_weights"][0]
axes[1, 0].bar([0], attention_sample, color="purple", label="Attention weight")
axes[1, 0].set_xticks([0])
axes[1, 0].set_xticklabels(["Single input position"])
axes[1, 0].set_ylim(0, 1.05)
axes[1, 0].set_ylabel("Weight")
axes[1, 0].set_title("Attention calculation in this implementation")
axes[1, 0].legend()
axes[1, 0].grid(True, axis="y", alpha=0.3)

metric_names = ["MAE", "MAPE", "RMSE"]
metric_keys = ["mae", "mape", "rmse"]
maxima = {
    key: max(float(result[key]) for result in methods_comparison)
    for key in metric_keys
}
for result in methods_comparison:
    values = [
        1.0 - float(result[key]) / maxima[key] if maxima[key] > 0 else 1.0
        for key in metric_keys
    ]
    axes[1, 1].plot(metric_names, values, marker="o", label=result["method"])
axes[1, 1].set_ylim(0, 1)
axes[1, 1].set_ylabel("Relative score (higher is better)")
axes[1, 1].set_title("Normalized error metrics")
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Interpreting the results and model limitations

Use the printed holdout metrics and plots to describe what happened in this particular run. A lower error on this holdout is evidence only about these 20 synthetic observations under this setup; it is not proof of consistent superiority. The training and validation curves help diagnose optimization behavior, but one split does not establish reliable generalization.

The auxiliary alpha value is constrained to the configured range for reporting and is regularized toward 0.3 during training. Because it does not enter the forecast equation, its variation should not be interpreted as adaptive exponential smoothing. Likewise, the attention weight is one because the implementation supplies a single sequence position. The interval has a fixed width derived from training-target variability; its observed holdout coverage is a descriptive diagnostic, not evidence of calibrated 95% coverage.

The model demonstrates a basic neural forecasting workflow: chronological window construction, training-only normalization, mini-batch gradient descent, validation monitoring, recursive prediction, and holdout evaluation. Its deliberately narrow scope makes the distinction between a model's implemented behavior and broader claims about attention, self-supervision, uncertainty quantification, and adaptive smoothing explicit.